In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

import healpy as hp
from astropy.coordinates import SkyCoord
from astropy import units

from healpy.newvisufunc import projview, newprojplot

sys.path.append(os.path.expanduser('~/git/desi-targets/dr9/density_variations/'))
from compute_weights import get_weights

In [2]:
hp.__version__

'1.15.0'

In [3]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [4]:
# Font sizes for healpix maps
fontsize_dict = {
    "xlabel": 9.5,
    "ylabel": 9.5,
    "title": 11.5,
    "xtick_label": 9.5,
    "ytick_label": 9.5,
    "cbar_label": 9.5,
    "cbar_tick_label": 9.5,
}

In [5]:
target_class = 'LRG'

target_ver_str = '1.0.0'

min_nobs = 1
# maskbits_dict = {'LRG': [1, 8, 9, 11, 12, 13], 'ELG': [1, 11, 12, 13], 'QSO': [1, 8, 9, 11, 12, 13], 'BGS_ANY': [1, 13], 'BGS_BRIGHT': [1, 13]}
maskbits_dict = {'LRG': [], 'ELG': [1, 11, 12, 13], 'QSO': [1, 8, 9, 11, 12, 13], 'BGS_ANY': [1, 13], 'BGS_BRIGHT': [1, 13]}
apply_lrgmask = True

if apply_lrgmask:
    lrgmask_str = '_lrgmask_v1'
else:
    lrgmask_str = ''

randoms_counts_dir = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/imaging_sys/randoms_stats/0.49.0/resolve/counts'
randoms_systematics_dir = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/imaging_sys/randoms_stats/0.49.0/resolve/systematics'
target_densities_dir = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/imaging_sys/density_maps/{}/resolve'.format(target_ver_str)

dpi_dict = {64: 200, 128: 200, 256: 600, 512: 1600}
xsize_dict = {64: 8000, 128: 8000, 256: 12000, 512: 16000}
vrange_dict = {'LRG': {64: [300, 900], 128: [200, 1000], 256: [100, 1100], 512: [-100, 1300]}}
# vrange_dict = {64: [0, 1200], 128: [-200, 1400], 256: [-600, 1800]}

min_pix_frac = 0.2  # minimum fraction of pixel area to be used

plt.rcParams['image.cmap'] = 'jet'

In [6]:
weights_path = '/Users/rongpu/Documents/Data/desi_targets/dr9.0/imaging_sys/linear_weights/main_v0.2/main_lrg_linear_coeffs_v0.2.yaml'

print(target_class)
target_class = target_class.lower()

maskbits = maskbits_dict[target_class.upper()]

nside = 256

npix = hp.nside2npix(nside)
pix_area = hp.pixelfunc.nside2pixarea(nside, degrees=True)
print(nside, 'Healpix size = {:.5f} sq deg'.format(pix_area))

field = 'combined'

for field in ['north', 'south']:

    if field=='north' or field=='BASS+MzLS':
        photsys = 'N'
    else:
        photsys = 'S'

    density = Table.read(os.path.join(target_densities_dir, 'density_map_{}_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(target_class, field, nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
    maps = Table.read(os.path.join(randoms_counts_dir, 'counts_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(field, nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
    maps = maps[maps['n_randoms']>0]
    maps1 = Table.read(os.path.join(randoms_systematics_dir, 'systematics_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(field, nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
    maps1.remove_columns(['RA', 'DEC'])
    maps = join(maps, maps1, join_type='inner', keys='HPXPIXEL')
    maps = join(maps, density[['HPXPIXEL', 'n_targets']], join_type='outer', keys='HPXPIXEL').filled(0)

    # # Load stellar density map
    # stardens = np.load('/Users/rongpu/Documents/Data/desi_lrg_selection/dr7/healpix_maps/pixweight-dr7.1-0.22.0_stardens_{}_ring.npy'.format(nside))
    # maps['stardens'] = stardens[maps['HPXPIXEL']]
    # maps['stardens_log'] = np.log10(maps['stardens'])

    maps['PHOTSYS'] = photsys

    maps['density_predict'] = get_weights(maps, weights_path, separate_des=False)
    maps['n_targets_predict'] = maps['density_predict'] * (pix_area * maps['FRACAREA'])

    if field=='north':
        maps_north = maps.copy()
    else:
        maps_south = maps.copy()

########## Combine the two maps; proper handling of overlapping pixels ##########

pix_overlap = np.intersect1d(maps_north['HPXPIXEL'], maps_south['HPXPIXEL'])
mask = np.in1d(maps_north['HPXPIXEL'], pix_overlap)
maps_overlap_north = maps_north[mask]
maps_north = maps_north[~mask]
mask = np.in1d(maps_south['HPXPIXEL'], pix_overlap)
maps_overlap_south = maps_south[mask]
maps_south = maps_south[~mask]

maps_overlap_north.sort('HPXPIXEL')
maps_overlap_south.sort('HPXPIXEL')

maps_overlap = maps_overlap_south.copy()
maps_overlap['n_targets'] = maps_overlap_north['n_targets'] + maps_overlap_south['n_targets']
maps_overlap['n_targets_predict'] = maps_overlap_north['n_targets_predict'] + maps_overlap_south['n_targets_predict']
maps_overlap['FRACAREA'] = maps_overlap_north['FRACAREA'] + maps_overlap_south['FRACAREA']
maps_overlap['density_predict'] = maps_overlap['n_targets_predict'] / (pix_area * maps_overlap['FRACAREA'])

maps = vstack([maps_north, maps_south, maps_overlap])

######################################################################

print(len(maps))

area = np.sum(maps['FRACAREA'])*pix_area
print('Area = {:.1f} sq deg'.format(area))

mask = maps['FRACAREA']>min_pix_frac
maps = maps[mask]

maps['density'] = maps['n_targets'] / (pix_area * maps['FRACAREA'])

LRG
256 Healpix size = 0.05246 sq deg
381901
Area = 17996.3 sq deg


In [7]:
# Galactic plane
org = 120
tmpn = 1000
cs = SkyCoord(l = np.linspace(0, 360, tmpn) * units.deg, b = np.zeros(tmpn) * units.deg, frame = "galactic")
ras, decs = cs.icrs.ra.degree, cs.icrs.dec.degree
ras = np.remainder(ras + 360 - org, 360) # shift ra values
ras[ras > 180] -= 360 # scale conversion to [-180, 180]
ii = ras.argsort()
ras, decs = ras[ii], decs[ii]

In [8]:
# Density map
map_values = np.zeros(npix)
hp_mask = np.zeros(npix, dtype=bool)
map_values[maps['HPXPIXEL']] = maps['density']
hp_mask[maps['HPXPIXEL']] = True
mplot = hp.ma(map_values)
mplot.mask = ~hp_mask

In [9]:
projview(mplot, min=vrange_dict[target_class.upper()][nside][0], max=vrange_dict[target_class.upper()][nside][1],
         rot=(120, 0, 0), cmap='jet', xsize=12000,
         graticule=True, graticule_labels=True, projection_type="mollweide",
         # title='{} NSIDE={}'.format(target_class.upper(), nside),
         xlabel='RA (deg)', ylabel='Dec (deg)',
         custom_xtick_labels=[r'$240\degree$', r'$180\degree$', r'$120\degree$', r'$60\degree$', r'$0\degree$'],
         fontsize=fontsize_dict)
newprojplot(theta=np.radians(90-decs), phi=np.radians(ras), color='k', lw=1)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/density_lrg_{}_lrgmask.png".format(nside), bbox_inches="tight", dpi=600)
plt.close()

In [10]:
# Weighted density map
map_values = np.zeros(npix)
hp_mask = np.zeros(npix, dtype=bool)
map_values[maps['HPXPIXEL']] = maps['density']/maps['density_predict']
hp_mask[maps['HPXPIXEL']] = True
mplot_weighted = hp.ma(map_values)
mplot_weighted.mask = ~hp_mask

In [11]:
projview(mplot_weighted, min=1./6, max=11./6,
         rot=(120, 0, 0), cmap='jet', xsize=12000,
         graticule=True, graticule_labels=True, projection_type="mollweide",
         # title='{} NSIDE={}'.format(target_class.upper(), nside),
         xlabel='RA (deg)', ylabel='Dec (deg)',
         custom_xtick_labels=[r'$240\degree$', r'$180\degree$', r'$120\degree$', r'$60\degree$', r'$0\degree$'],
         fontsize=fontsize_dict)
newprojplot(theta=np.radians(90-decs), phi=np.radians(ras), color='k', lw=1)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/more/density_lrg_{}_lrgmask_weighted.png".format(nside), bbox_inches="tight", dpi=600)
plt.close()

In [12]:
# Weight map
map_values = np.zeros(npix)
hp_mask = np.zeros(npix, dtype=bool)
map_values[maps['HPXPIXEL']] = maps['density_predict']
hp_mask[maps['HPXPIXEL']] = True
mplot_weights = hp.ma(map_values)
mplot_weights.mask = ~hp_mask

In [13]:
projview(mplot_weights, min=500, max=700,
         rot=(120, 0, 0), cmap='jet', xsize=12000,
         graticule=True, graticule_labels=True, projection_type="mollweide",
         # title='{} NSIDE={}'.format(target_class.upper(), nside),
         xlabel='RA (deg)', ylabel='Dec (deg)',
         custom_xtick_labels=[r'$240\degree$', r'$180\degree$', r'$120\degree$', r'$60\degree$', r'$0\degree$'],
         fontsize=fontsize_dict)
newprojplot(theta=np.radians(90-decs), phi=np.radians(ras), color='k', lw=1)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/more/density_lrg_{}_lrgmask_weight_map.png".format(nside), bbox_inches="tight", dpi=600)
plt.close()

__Variations of the density map on with different resolution or color stretch__

In [14]:
projview(mplot, min=vrange_dict[target_class.upper()][nside][0], max=vrange_dict[target_class.upper()][nside][1],
         rot=(120, 0, 0), cmap='jet', xsize=6000,
         graticule=True, graticule_labels=True, projection_type="mollweide",
         # title='{} NSIDE={}'.format(target_class.upper(), nside),
         xlabel='RA (deg)', ylabel='Dec (deg)',
         custom_xtick_labels=[r'$240\degree$', r'$180\degree$', r'$120\degree$', r'$60\degree$', r'$0\degree$'],
         fontsize=fontsize_dict)
newprojplot(theta=np.radians(90-decs), phi=np.radians(ras), color='k', lw=1)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/more/density_lrg_{}_lrgmask_1.png".format(nside), bbox_inches="tight", dpi=200)
plt.close()

In [15]:
projview(mplot, min=0, max=1200,
         rot=(120, 0, 0), cmap='jet', xsize=12000,
         graticule=True, graticule_labels=True, projection_type="mollweide",
         # title='{} NSIDE={}'.format(target_class.upper(), nside),
         xlabel='RA (deg)', ylabel='Dec (deg)',
         custom_xtick_labels=[r'$240\degree$', r'$180\degree$', r'$120\degree$', r'$60\degree$', r'$0\degree$'],
         fontsize=fontsize_dict)
newprojplot(theta=np.radians(90-decs), phi=np.radians(ras), color='k', lw=1)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/more/density_lrg_{}_lrgmask_2.png".format(nside), bbox_inches="tight", dpi=600)
plt.close()

In [16]:
projview(mplot, min=vrange_dict[target_class.upper()][nside][0], max=vrange_dict[target_class.upper()][nside][1],
         rot=(120, 0, 0), cmap='jet', xsize=12000,
         graticule=True, graticule_labels=True, projection_type="mollweide",
         # title='{} NSIDE={}'.format(target_class.upper(), nside),
         xlabel='RA (deg)', ylabel='Dec (deg)',
         custom_xtick_labels=[r'$240\degree$', r'$180\degree$', r'$120\degree$', r'$60\degree$', r'$0\degree$'],
         fontsize=fontsize_dict)
newprojplot(theta=np.radians(90-decs), phi=np.radians(ras), color='k', lw=1)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/more/density_lrg_{}_lrgmask_3.png".format(nside), bbox_inches="tight", dpi=450)
plt.close()

-------
__Density map of all LRG targets (i.e., no masking beyond the target selection pipeline)__

In [17]:
min_nobs = 1
maskbits = [1, 12, 13]
lrgmask_str = ''

In [18]:
print(target_class)
target_class = target_class.lower()

nside = 256

npix = hp.nside2npix(nside)
pix_area = hp.pixelfunc.nside2pixarea(nside, degrees=True)
print(nside, 'Healpix size = {:.5f} sq deg'.format(pix_area))

field = 'combined'

for field in ['north', 'south']:

    if field=='north' or field=='BASS+MzLS':
        photsys = 'N'
    else:
        photsys = 'S'

    density = Table.read(os.path.join(target_densities_dir, 'density_map_{}_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(target_class, field, nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
    maps = Table.read(os.path.join(randoms_counts_dir, 'counts_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(field, nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
    maps = maps[maps['n_randoms']>0]
    # maps1 = Table.read(os.path.join(randoms_systematics_dir, 'systematics_{}_nside_{}_minobs_{}_maskbits_{}.fits'.format(field, nside, min_nobs, ''.join([str(tmp) for tmp in maskbits])+lrgmask_str)))
    # maps1.remove_columns(['RA', 'DEC'])
    # maps = join(maps, maps1, join_type='inner', keys='HPXPIXEL')
    maps = join(maps, density[['HPXPIXEL', 'n_targets']], join_type='outer', keys='HPXPIXEL').filled(0)

    # # Load stellar density map
    # stardens = np.load('/Users/rongpu/Documents/Data/desi_lrg_selection/dr7/healpix_maps/pixweight-dr7.1-0.22.0_stardens_{}_ring.npy'.format(nside))
    # maps['stardens'] = stardens[maps['HPXPIXEL']]
    # maps['stardens_log'] = np.log10(maps['stardens'])

    maps['PHOTSYS'] = photsys

    if field=='north':
        maps_north = maps.copy()
    else:
        maps_south = maps.copy()

########## Combine the two maps; proper handling of overlapping pixels ##########

pix_overlap = np.intersect1d(maps_north['HPXPIXEL'], maps_south['HPXPIXEL'])
mask = np.in1d(maps_north['HPXPIXEL'], pix_overlap)
maps_overlap_north = maps_north[mask]
maps_north = maps_north[~mask]
mask = np.in1d(maps_south['HPXPIXEL'], pix_overlap)
maps_overlap_south = maps_south[mask]
maps_south = maps_south[~mask]

maps_overlap_north.sort('HPXPIXEL')
maps_overlap_south.sort('HPXPIXEL')

maps_overlap = maps_overlap_south.copy()
maps_overlap['n_targets'] = maps_overlap_north['n_targets'] + maps_overlap_south['n_targets']
maps_overlap['FRACAREA'] = maps_overlap_north['FRACAREA'] + maps_overlap_south['FRACAREA']

maps = vstack([maps_north, maps_south, maps_overlap])

######################################################################

print(len(maps))

area = np.sum(maps['FRACAREA'])*pix_area
print('Area = {:.1f} sq deg'.format(area))

mask = maps['FRACAREA']>min_pix_frac
maps = maps[mask]

maps['density'] = maps['n_targets'] / (pix_area * maps['FRACAREA'])

lrg
256 Healpix size = 0.05246 sq deg
382087
Area = 19517.0 sq deg


In [19]:
# Density map
map_values = np.zeros(npix)
hp_mask = np.zeros(npix, dtype=bool)
map_values[maps['HPXPIXEL']] = maps['density']
hp_mask[maps['HPXPIXEL']] = True
mplot_nomask = hp.ma(map_values)
mplot_nomask.mask = ~hp_mask

In [20]:
projview(mplot_nomask, min=vrange_dict[target_class.upper()][nside][0], max=vrange_dict[target_class.upper()][nside][1],
         rot=(120, 0, 0), cmap='jet', xsize=12000,
         graticule=True, graticule_labels=True, projection_type="mollweide",
         # title='{} NSIDE={}'.format(target_class.upper(), nside),
         xlabel='RA (deg)', ylabel='Dec (deg)',
         custom_xtick_labels=[r'$240\degree$', r'$180\degree$', r'$120\degree$', r'$60\degree$', r'$0\degree$'],
         fontsize=fontsize_dict)
newprojplot(theta=np.radians(90-decs), phi=np.radians(ras), color='k', lw=1)
plt.savefig("/Users/rongpu/git/desi-targets/lrg_paper/figures/more/density_lrg_{}_nomask.png".format(nside), bbox_inches="tight", dpi=600)
plt.close()